In [1]:
import numpy as np
# 本质上线性变换就只有乘和加两种运算
class MulLayer():
    def __init__(self,):
        # 乘法层的导数是两个相乘变量互相交换
        # 所以必须记录x，y前向传播的输入是啥
        # f(z) = xy
        self.x = None
        self.y = None
    
    def forward(self,x,y):
        self.x = x
        self.y = y
        return x*y

    def backward(self,dz):
        dx = dz*self.y
        dy = dz*self.x
        return dx,dy

apple = 100
nums = 2
tax = 1.1

layer1 = MulLayer()
l1_out = layer1.forward(apple,nums)
layer2 = MulLayer()
l2_out = layer2.forward(l1_out,tax)

print(l2_out)

# 反向传播过来的首个偏导数，因为没有其他的输入，只有这一个
# 所以f(z) = z，而且是在全连接参数w的位置，不是神经元的位置
dprice = 1
dapple_total_price,d_tax = layer2.backward(dprice)
d_apple_price,d_apple_nums = layer1.backward(dapple_total_price)
print(d_apple_price,d_apple_nums,d_tax)

220.00000000000003
2.2 110.00000000000001 200


In [2]:
# 加法节点反向传播
class AddLayer():
    def __init__(self,):
        pass

    def forward(self,x,y):
        return x+y

    def backward(self,dz):
        # 直接传递下去就行
        return dz*1,dz*1

# 通过乘加实现买苹果的例子
apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1

# 3个乘法层，1个加法层
mul1 = MulLayer()
mul2 = MulLayer()
mul3 = MulLayer()
add1 = AddLayer()

# 前向传播
mul1_out = mul1.forward(apple_num,apple)
mul2_out = mul2.forward(orange_num,orange)
add1_out = add1.forward(mul1_out,mul2_out)
mul3_out = mul3.forward(add1_out,tax)
print(mul3_out)

# 反向传播
mul3_back_out_add1,mul3_back_out_tax = mul3.backward(1) 
add1_back_out_mul1,add1_back_out_mul2 = add1.backward(mul3_back_out_add1)
mul2_back_out_orange_num,mul2_back_out_orange = mul2.backward(add1_back_out_mul2)
mul1_back_out_apple_num,mul1_back_out_apple = mul1.backward(add1_back_out_mul1)
print(mul1_back_out_apple,mul1_back_out_apple_num,mul2_back_out_orange,mul2_back_out_orange_num,mul3_back_out_tax)

715.0000000000001
2.2 110.00000000000001 3.3000000000000003 165.0 650


In [3]:
# relu激活函数前/反向传播层
# 鱼书在forward中用了copy，其实没必要，形式参数不会改变原来张量的值，用copy主要是因为前向激活值被改了反向传播就不对了
# 那么啥时候可以原地修改，啥时候必须要新建/缓存下来呢？
# 理论上每个前向传播层都是要记录原始的激活值（激活值的意思是每层的输出）
# 因为反向传播用得到，比如前面的z=xy，不乘的话就不知道怎么把导数传播下去了
# 像relu/加法不用存的原因是因为偏导数是1
"""
Level 0: 全部新建（最安全，最费显存）
  ↓
Level 1: 反向传播中原地修改梯度（几乎无风险）
  ↓
Level 2: ReLU 等用 inplace=True（小心使用）
  ↓
Level 3: Gradient Checkpointing（前向传播不存中间值，
         反向传播时重新算一遍，用时间换显存）
  ↓
Level 4: 混合精度训练（FP16 前向/反向，FP32 存权重）
"""
class ReluLayer():
    def __init__(self,):
        self.mask = None

    def forward(self,x):
        # <=的运算符号重载，直接返回一个true/false的矩阵
        # eg：[1,2,0,4,5]->[False,False,True,False,False,]
        self.mask = (x<=0)
        return np.maximum(x,0)

    def backward(self,dz):
        # 对应的下标位置为0，其他的都是原封不动传下去
        # 布尔索引，先取出所有为True的下标，然后基于花式索引赋值
        # eg: [1,2,0,4,5]->[False,False,True,False,False,] ->[3](x[self.mask]等价于x[[3]])
        dz[self.mask] = 0
        return dz


In [6]:
# sigmoid layer
class SigmoidLayer():
    def __init__(self,):
        # 这里本质上存的也是input，只是sigmoid
        self.out = None

    def forward(self,x):
        out = 1/(np.exp(-1*x)+1)
        self.out = out
        return out

    def backward(self,dz):
        # f'(x) = f(x)(1-f'(x))
        dx = dz*self.out*(1-self.out)
        return dx

# linear layer
# 对应书上的affline仿射变换层
# 这里有两点要注意：
# 1.XW求偏导的时候需要保证矩阵形状一致，并且注意转置矩阵的位置
# 2.若是批处理的偏置，则虽然偏置不需要[N,b]，应该是偏置在反向传播的时候应该汇总所有偏置bias值
# 问题：bias为什么不是求平均？->loss函数求过了不求，反之就需要求。
class LinearLayer():
    def __init__(self,w,b):
        # 可变对象与外部共享同一片内存（按引用传递list, dict, np.ndarray，
        # 但是如果是不可变对象则为按值传递int, float, str, tuple）
        self.w = w
        self.b = b
        self.x = None
        # 为什么 dw、db 不直接返回，而是存在 self 里？
        # backward 返回的 dx 是要继续往前一层传的梯度，是"链式法则的接力棒"。反向传播和权重更新是分开的两个阶段
        # dw 和 db 是这一层自己的参数梯度，用于更新权重，不需要传给前一层。（用于优化器）
        self.dw = None
        self.db = None


    def forward(self,x):
        self.x = x
        # 设w shape [n,m],b shape[1,m],x shape [N,n]
        # out shape [N,m]
        out = np.dot(x,self.w) + self.b
        return out

    def backward(self,dz):
        # dz shape [N,m],wT shape[m,n]-> dx shape [N,n]
        dx = np.dot(dz,self.w.T)
        # dz shape [N,m],xT shape[n,N]-> dx shape [n,m]
        self.dw = np.dot(self.x.T,dz)
        # bias是一个加法，传播过来就是dz，但是要求和[N,m]
        self.db = np.sum(dz,axis = 0)
        return dx

In [ ]:
# softmax+交叉熵函数
# 这里的计算图很复杂，但是一步步还是能推导下来的。
# 其中有一点，就是在计算图多分支的时候，反向传播先求和再反向传播求偏导。
# 易得出，softmax+交叉熵，反向传播回来的损失是y_pred-y_true这个值。
# 这个是构造的组合，假设数据服从某种分布 → 最大似然估计"自然推导出来的。
# 选什么激活函数由分布的值域决定（正数用 exp，概率用 sigmoid/softmax），
# 选什么损失函数由分布的对数似然决定。它们的梯度都会简化为"预测减真实"的优美形式。（参见正则链接函数）
class SoftmaxWithLoss():
    def __init__(self,):
        self.x = None
        self.y_true = None
        self.loss = None
    def forward(self,x,y_true):
        self.y_true = y_true
        self.x = softmax(x)
        self.loss = cross_entropy_loss(y_true = y_true,y_pred = self.x)

        return self.loss

    def backward(self,dz=1):
        batch = self.y_true.shape[0]
        return (self.x-self.y_true)/batch
